Khai báo thư viện

In [79]:
import pandas as pd
from sklearn.impute import SimpleImputer

train = pd.read_csv("./data/train.csv")
test = pd.read_csv("./data/test.csv")

print(train.head())


   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name     Sex   Age  SibSp  \
0                            Braund, Mr. Owen Harris    male  22.0      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                             Heikkinen, Miss. Laina  female  26.0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                           Allen, Mr. William Henry    male  35.0      0   

   Parch            Ticket     Fare Cabin Embarked  
0      0         A/5 21171   7.2500   NaN        S  
1      0          PC 17599  71.2833   C85        C  
2      0  STON/O2. 3101282   7.9250   NaN        S  
3      0            113803  53.1000  C123        S  
4      0            373450   8.0500   NaN        S  


Chọn features và labels

In [80]:
X = train[['Pclass', 'Sex', 'Age', 'Fare', 'SibSp', 'Parch', 'Embarked']]
y = train['Survived']



Chuyển đổi chữ sang số

In [81]:
num_cols = X.select_dtypes(include=['int64', 'float64']).columns
cat_cols = X.select_dtypes(include=['object']).columns

imputer = SimpleImputer(strategy='mean')
X[num_cols] = imputer.fit_transform(X[num_cols])

X[cat_cols] = X[cat_cols].fillna('missing')

for col in cat_cols:
    X[col] = pd.factorize(X[col])[0]

C:\Users\LE DINH\AppData\Local\Temp\ipykernel_17144\3411822210.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X[num_cols] = imputer.fit_transform(X[num_cols])
C:\Users\LE DINH\AppData\Local\Temp\ipykernel_17144\3411822210.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X[cat_cols] = X[cat_cols].fillna('missing')
C:\Users\LE DINH\AppData\Local\Temp\ipykernel_17144\3411822210.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,

Chia train/test

In [82]:
from sklearn.model_selection import train_test_split

X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, random_state=42)



Train model

In [83]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier


logis = LogisticRegression(max_iter=1000, random_state=42)
logis.fit(X_train, y_train)
y_pred_logis = logis.predict(X_valid)

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_valid)

knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)
y_pred_knn = knn.predict(X_valid)

dt = DecisionTreeClassifier(random_state=42)
dt.fit(X_train, y_train)
y_pred_dt = dt.predict(X_valid)

xgb = XGBClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=4,
    random_state=42,
    use_label_encoder=False,
    eval_metric='logloss'
)
xgb.fit(X_train, y_train)
y_pred_xgb = xgb.predict(X_valid)

import pandas as pd

results = []
models = {
    "Decision Tree": y_pred_dt,
    "Logistic Regression": y_pred_logis,
    "Random Forest": y_pred_rf,
    "KNN": y_pred_knn,
    "XGBoost": y_pred_xgb
}

for name, y_pred in models.items():
    acc = accuracy_score(y_valid, y_pred)
    pre = precision_score(y_valid, y_pred)
    rec = recall_score(y_valid, y_pred)
    f1 = f1_score(y_valid, y_pred)

    results.append({
        "Model": name,
        "Accuracy": round(acc, 4),
        "Precision": round(pre, 4),
        "Recall": round(rec, 4),
        "F1-Score": round(f1, 4)
    })

df_results = pd.DataFrame(results)
print(df_results.sort_values(by="F1-Score", ascending=False).reset_index(drop=True))

                 Model  Accuracy  Precision  Recall  F1-Score
0        Random Forest    0.8268     0.8116  0.7568    0.7832
1              XGBoost    0.8156     0.8154  0.7162    0.7626
2  Logistic Regression    0.7989     0.7714  0.7297    0.7500
3        Decision Tree    0.7821     0.7465  0.7162    0.7310
4                  KNN    0.7039     0.6780  0.5405    0.6015


c:\Users\LE DINH\anaconda3\Lib\site-packages\xgboost\training.py:183: UserWarning: [23:03:06] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Thêm 3 models mới

In [84]:
from sklearn.svm import SVC
from sklearn.ensemble import GradientBoostingClassifier, AdaBoostClassifier

svc = SVC(probability=True, random_state=42)
svc.fit(X_train, y_train)
y_pred_svc = svc.predict(X_valid)

gb = GradientBoostingClassifier(random_state=42)
gb.fit(X_train, y_train)
y_pred_gb = gb.predict(X_valid)

ada = AdaBoostClassifier(random_state=42)
ada.fit(X_train, y_train)
y_pred_ada = ada.predict(X_valid)

extra_models = {
    "SVC": y_pred_svc,
    "Gradient Boosting": y_pred_gb,
    "AdaBoost": y_pred_ada
}

for name, y_pred in extra_models.items():
    acc = accuracy_score(y_valid, y_pred)
    pre = precision_score(y_valid, y_pred)
    rec = recall_score(y_valid, y_pred)
    f1 = f1_score(y_valid, y_pred)
    
    results.append({
        "Model": name,
        "Accuracy": round(acc, 4),
        "Precision": round(pre, 4),
        "Recall": round(rec, 4),
        "F1-Score": round(f1, 4)
    })

df_results = pd.DataFrame(results)
df_results = df_results.sort_values(by="F1-Score", ascending=False).reset_index(drop=True)
print(df_results)


                 Model  Accuracy  Precision  Recall  F1-Score
0        Random Forest    0.8268     0.8116  0.7568    0.7832
1              XGBoost    0.8156     0.8154  0.7162    0.7626
2    Gradient Boosting    0.8156     0.8361  0.6892    0.7556
3  Logistic Regression    0.7989     0.7714  0.7297    0.7500
4             AdaBoost    0.7933     0.7606  0.7297    0.7448
5        Decision Tree    0.7821     0.7465  0.7162    0.7310
6                  KNN    0.7039     0.6780  0.5405    0.6015
7                  SVC    0.6592     0.7600  0.2568    0.3838


Chọn ra 3 model cho ra kết quả tốt nhất

In [85]:
top3_models = df_results.head(3)["Model"].tolist()
print("\n3 model tốt nhất:", top3_models)



3 model tốt nhất: ['Random Forest', 'XGBoost', 'Gradient Boosting']


Thêm tham số cho 3 model tốt nhất

In [86]:
from sklearn.model_selection import GridSearchCV

param_grids = {
    "Random Forest": {
        'n_estimators': [100, 200, 300],
        'max_depth': [4, 6, 8, None]
    },
    "XGBoost": {
        'n_estimators': [100, 200],
        'max_depth': [3, 4, 5],
        'learning_rate': [0.05, 0.1]
    },
    "Gradient Boosting": {
        'n_estimators': [100, 200],
        'learning_rate': [0.05, 0.1],
        'max_depth': [3, 4, 5]
    }
}

best_models = {}
for model_name in top3_models:
    print(f"\nTuning {model_name}...")
    if model_name not in param_grids:
        print(f"Bỏ qua {model_name} (chưa định nghĩa grid search)")
        continue
    
    model_obj = {
        "Random Forest": rf,
        "XGBoost": xgb,
        "Gradient Boosting": gb
    }[model_name]
    
    grid = GridSearchCV(model_obj, param_grids[model_name], cv=5, scoring='f1', n_jobs=-1)
    grid.fit(X_train, y_train)
    best_models[model_name] = grid.best_estimator_
    print(f"Best params for {model_name}: {grid.best_params_}")



Tuning Random Forest...
Best params for Random Forest: {'max_depth': 8, 'n_estimators': 300}

Tuning XGBoost...


c:\Users\LE DINH\anaconda3\Lib\site-packages\xgboost\training.py:183: UserWarning: [23:03:14] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Best params for XGBoost: {'learning_rate': 0.05, 'max_depth': 4, 'n_estimators': 200}

Tuning Gradient Boosting...
Best params for Gradient Boosting: {'learning_rate': 0.1, 'max_depth': 4, 'n_estimators': 100}


Chọn model tốt nhất

In [87]:
final_scores = {}
for name, model in best_models.items():
    y_pred = model.predict(X_valid)
    final_scores[name] = f1_score(y_valid, y_pred)

best_model_name = max(final_scores, key=final_scores.get)
best_model = best_models[best_model_name]
print(f"\nModel tốt nhất là: {best_model_name} (F1 = {final_scores[best_model_name]:.4f})")



Model tốt nhất là: Gradient Boosting (F1 = 0.7746)


Lấy model tốt nhất đi dự đoán 

In [88]:
test_data = pd.read_csv("./data/test.csv")
test_data = test_data[['Pclass', 'Name', 'Sex', 'Age', 'Fare', 'SibSp', 'Parch', 'Embarked']]
test_data = test_data.fillna(test_data.mean(numeric_only=True))
test_data = test_data.fillna('missing')

test_data = pd.get_dummies(test_data)
missing_cols = set(X_train.columns) - set(test_data.columns)
for c in missing_cols:
    test_data[c] = 0
test_data = test_data[X_train.columns]

y_test_pred = best_model.predict(test_data)

submission = pd.DataFrame({
    "PassengerId": pd.read_csv("./data/test.csv")["PassengerId"],
    "Survived": y_test_pred
})
submission.to_csv("submission.csv", index=False)
print("\n Đã tạo file submission.csv")



 Đã tạo file submission.csv
